# Linear Regression in Python

---

## 1. Introduction 

This notebook demonstrates fitting linear regression models in a Jupyter notebook using Python libraries and NHANES data. NHANES is a complex, weighted survey (with strata and clusters) that ideally requires survey-aware methods, but here its design will be ignored to illustrate regression on independent or convenience samples.

We will focus initially on regression models in which systolic [blood pressure](https://en.wikipedia.org/wiki/Blood_pressure) (SBP) is the outcome (dependent) variable. That is, we will predict a subject's SBP from other variables relating to that subject. 

SBP is an important indicator of cardiovascular health. It tends to increase with age, is greater for overweight people (i.e. people with greater body mass index or BMI), and also differs among demographic groups, for example among gender and ethnic groups. 

Here we will model SBP using linear regression because linear regression is a good default starting point for any regression analysis using a quantitative outcome variable.

---

## 2. Import Libraries and Load Data
First, we will import the necessary libraries for data manipulation, visualization, and statistical modeling. Then we will load the data.

The NHANES study encompasses multiple waves of data collection. Here we only use the 2015-2016 data. As with most data sets, there are some missing values in the NHANES files. For simplicity, we'll explicitly drop all observations with missing values in any of the key variables that we will use in this notebook. This is called *complete case analysis*.

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Read the 2015-2016 wave of NHANES data
nhanes_df = pd.read_csv("./data/nhanes_2015_2016.csv")

# Drop unused columns, and drop rows with any missing values.
vars = ["BPXSY1", "RIDAGEYR", "RIAGENDR", "RIDRETH1", "DMDEDUC2", "BMXBMI", "SMQ020"]
nhanes_df = nhanes_df[vars].dropna()


---

## 3. Fitting a Simple Linear Regression Model
We start with a **simple linear regression model** which has only one covariate, age, predicting SBP.  In the NHANES data, the variable [BPXSY1](https://wwwn.cdc.gov/Nchs/Nhanes/2015-2016/BPX_I.htm#BPXSY1) contains the first recorded measurement of SBP for a subject, and [RIDAGEYR](https://wwwn.cdc.gov/Nchs/Nhanes/2015-2016/DEMO_I.htm#RIDAGEYR) is the subject's age in years.  The model that is fit in the next cell expresses the expected value of SBP as a linear function of age.  The formula `BPXSY1 ~ RIDAGEYR` indicates that the variable named `BPXSY1` is the response variable in this regression analysis, and the analysis has one covariate, which is `RIDAGEYR`.

In [21]:
model = sm.OLS.from_formula("BPXSY1 ~ RIDAGEYR", data=nhanes_df)
result = model.fit()
print(result.summary())
print("="*78)

                            OLS Regression Results                            
Dep. Variable:                 BPXSY1   R-squared:                       0.207
Model:                            OLS   Adj. R-squared:                  0.207
Method:                 Least Squares   F-statistic:                     1333.
Date:                Wed, 22 Oct 2025   Prob (F-statistic):          2.09e-259
Time:                        20:06:54   Log-Likelihood:                -21530.
No. Observations:                5102   AIC:                         4.306e+04
Df Residuals:                    5100   BIC:                         4.308e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    102.0935      0.685    149.120      0.0

### Interpretting Regression Parameters

At the moment, we will focus on the center section of the output where the header row begins with **coef**.  This section contains the estimated values of the parameters of the regression model, their standard errors, and other values that are used to quantify the uncertainty in the regression parameter estimates.  Note that the regression parameters may also be referred to as *slopes* or *effects*.

This fitted model implies that when comparing two people whose ages differ by one year, the older person will on average have 0.48 units higher SBP than the younger person. This difference is statistically significant, based on the p-value shown under the column labeled __`P>|t|`__.  This means that there is strong evidence that there is a real **association** between between systolic blood pressure and age in this population.

SBP is measured in units of *millimeters of mercury*, expressed *mm/Hg*.  In order to better understand the meaning of the estimated regression parameter 0.48, we can look at the standard deviation of SBP:

In [22]:
print(nhanes_df["BPXSY1"].std())

18.486559500782416


The standard deviation of around 18.5 describes the *unexplained variation* in systolic blood pressure values.  It is large compared to the regression slope of 0.48, which describes the average difference between blood pressure values for two people whose ages differ by one year. Thus, while there is a substantial tendency for blood pressure to increase with age, there is also a great deal of variation among people with the same age -- we should not be surprised to find, say, a 40 year old with greater blood pressure than a 60 year old.

### R-squared vs Correlation for a Simple Linear Regression Model
In the case of regression with a single independent variable, as we have here, there is a very close correspondence between the regression analysis and a Pearson correlation analysis.

The primary summary statistic for assessing the strength of a predictive relationship in a linear regression model is the *R-squared*, which is shown to be 0.207 in the regression output above.  This means that 21% of the variation in SBP is explained by age.  Note that, for a simple linear regression model, this value is always exactly the same as the squared Pearson correlation coefficient between SBP and age, as shown below.

In [23]:
cc = nhanes_df[["BPXSY1", "RIDAGEYR"]].corr()
print(cc.BPXSY1.RIDAGEYR**2)

0.2071545962518702


There is a second way to interpret the R-squared, which makes use
of the *fitted values* of the regression.  The fitted values are
predictions of the blood pressure for each person in the data
set, based on their covariate values.  In this example, the only
covariate is age, so we are predicting each NHANES subject's
blood pressure as a function of their age.  If we calculate
the Pearson correlation coefficient between the fitted values
from the regression, and the actual SBP values, and then square
this correlation coefficient, we see
that we also get the R-squared from the regression:

In [24]:
cc = np.corrcoef(nhanes_df.BPXSY1, result.fittedvalues)
print(cc[0, 1]**2)

0.20715459625186938


Thus, we see that in a linear model fit with only one covariate (a.k.a. a simple linear regression model),
the regression R-squared is equal to the squared Pearson
correlation between the covariate and the outcome, and is also
equal to the squared Pearson correlation between the fitted
values and the outcome.

---

## 4. Adding a Second Variable to the Linear Model
Above we considered a simple linear regression analysis with only one covariate (age) predicting systolic blood pressure (SBP). The real power of regression analysis arises when we have more than one covariate predicting an outcome. This type of model is called **multiple linear regression**.

As noted above, SBP is expected to be related to gender as well as to age, so we next add gender to the model. The NHANES variable for gender is named `RIAGENDR`.

In [25]:
# Create a labeled version of the gender variable
nhanes_df['RIAGENDRx'] = nhanes_df['RIAGENDR'].replace({1: "Male", 2: "Female"})

# Fit the multiple linear regression model
model = sm.OLS.from_formula("BPXSY1 ~ RIDAGEYR + RIAGENDRx", data=nhanes_df)
result = model.fit()
print(result.summary())
print("="*78)

                            OLS Regression Results                            
Dep. Variable:                 BPXSY1   R-squared:                       0.215
Model:                            OLS   Adj. R-squared:                  0.214
Method:                 Least Squares   F-statistic:                     697.4
Date:                Wed, 22 Oct 2025   Prob (F-statistic):          1.87e-268
Time:                        20:06:55   Log-Likelihood:                -21505.
No. Observations:                5102   AIC:                         4.302e+04
Df Residuals:                    5099   BIC:                         4.304e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept           100.6305      0.71

### Interpretting Regression Parameters
The model that was fit above uses both age and gender to explain the variation in SBP.  It finds that **two people with the same gender whose ages differ by one year tend to have blood pressure values differing by 0.47 units**.  Note that this is essentially the same age parameter (0.48) that we found above in the model based on age alone.  This model also shows us that comparing a male and a female person of the same age, the male will on average have 3.23 units greater SBP than the female.

It is very important to emphasize that the age coefficient of 0.47 is only meaningful when comparing two people of the same gender, and the gender coefficient of 3.23 is only meaningful when comparing two people of the same age.

Moreover, **these effects are additive**, meaning that if we compare, say, a 50 year old man to a 40 year old woman, the man's blood pressure will on average be around $3.23 + 10\cdot 0.47 = 7.93$ units higher, with the first term in this sum being attributable to gender, and the second term being attributable to age.

We noted above that the regression coefficient for age did not change by much when we added gender to the model.  It is important to note however that in general, the estimated coefficient of a variable in a regression model will change when other variables are added or removed.  The regression parameter for one variable, say gender, is guaranteed to be unchanged when adding or removing another variable, say age, **only when the gender and age variables are uncorrelated** in the dataset.

Below we confirm that gender and age are nearly uncorrelated in this data set (the correlation of around -0.02 is negligible).  Thus, it is expected that when we add gender to the model, the age coefficient is unaffected.

In [26]:
# Use the original, numerical version of the gender variable to calculate the correlation coef
nhanes_df[["RIDAGEYR", "RIAGENDR"]].corr()

,RIDAGEYR,RIAGENDR
RIDAGEYR,1.000000,-0.021398
RIAGENDR,-0.021398,1.000000


### Interpretting the R-squared Value
Observe that in the regression output shown above, an R-squared value of 0.215 is
listed.  Earlier we saw that for a model with only one covariate,
the R-squared from the regression could be defined in two different
ways, either as the squared correlation coefficient between the covariate and the outcome,
or as the squared correlation coefficient between the fitted values and the outcome.
When more than one covariate is in the model, only the second of these
two definitions continues to hold:

In [27]:
cc = np.corrcoef(nhanes_df["BPXSY1"], result.fittedvalues)
print(cc[0, 1]**2)

0.21478581086243761


When adding variables to a multiple linear regression model, **the R-squared can never decrease**.  It can either remain unchanged, or it can increase.  In this example, the model accounting only for age had an R-squared of 0.207, and the model accounting for both age and gender had an R-squared of 0.215.  On the basis of R-squared, gender seems to have play only a small role in explaining the variation in systolic blood pressure.  While sex differences are present here, they are small in relation to the differences explained by age.

### Categorical Variables and Reference Levels

In the model fit above, gender is a categorical variable, and only a coefficient for males is included in the regression output (i.e. there is no coefficient for females in the tables above).  Whenever a categorical variable is used as a covariate in a regression model, one level of the variable is omitted and is automatically given a coefficient of zero.  This level is called the *reference level* of the covariate.  Here, the female level of the gender variable is the reference level.  This does not mean that being a woman has no impact on blood pressure.  It simply means that we have written the model so that female blood pressure is the default, and the coefficient for males (3.23) shifts the blood pressure by a certain amount for males only.

We could alternatively have set 'male' to be the reference level, in which case males would be the default, and the female coefficient would have been around -3.23 (meaning that female blood pressure is 3.23 units lower than the male blood pressure, at a fixed age).

When using a categorical variable as a predictor in a regression model, it is recoded into "dummy variables" (also known as "indicator variables").  A dummy variable for a single level, say $a$, of a variable $x$, is a variable that is equal to $1$ when $x=a$ and is equal to $0$ when $x$ is not equal to $a$.  These dummy variables are all included in the regression model, to represent the variable that they are derived from.  Thus, if we have a categorical variable with say three levels, it will enter the model as two indicator variables, and two regression parameters will be estimated to explain the role of this variable in the regression model.

Statsmodels, like most software, will automatically recode a categorical variable into dummy variables, and will select a reference level (it is possible to override this choice, but we do not cover that here).  When interpreting the regression output, the level that is omitted should be seen as having a coefficient of 0, with a standard error of 0.  It is important to note that the selection of a reference level is arbitrary and does not imply any assumption or constraint
about the model, or about the population that it is intended to capture.

---

## 5. A Linear Model with Three Variables
Next we add a third variable, body mass index (BMI), to the model predicting SBP. [BMI](https://en.wikipedia.org/wiki/Body_mass_index) is a measure that is used to assess a person's weight in a way that is adjusted for their height.  Unlike the raw weight, BMI does not automatically increase with height. [BMXBMI](https://wwwn.cdc.gov/Nchs/Nhanes/2015-2016/BMX_I.htm#BMXBMI) is the NHANES variable containing the BMI value for each subject.

In [28]:
model = sm.OLS.from_formula("BPXSY1 ~ RIDAGEYR + BMXBMI + RIAGENDRx", data=nhanes_df)
result = model.fit()
print(result.summary())
print("="*78)

                            OLS Regression Results                            
Dep. Variable:                 BPXSY1   R-squared:                       0.228
Model:                            OLS   Adj. R-squared:                  0.228
Method:                 Least Squares   F-statistic:                     502.0
Date:                Wed, 22 Oct 2025   Prob (F-statistic):          8.54e-286
Time:                        20:06:55   Log-Likelihood:                -21461.
No. Observations:                5102   AIC:                         4.293e+04
Df Residuals:                    5098   BIC:                         4.296e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            91.5840      1.19

Not surprisingly, BMI is positively associated with SBP.  Given two
subjects with the same gender and age, and whose BMI differs by 1
unit, the person with greater BMI will have, on average, 0.31 units
greater systolic blood pressure (SBP).  Also note that after adding
BMI to the model, the coefficient for gender became somewhat greater.
This is due to the fact that the three covariates in the model, age,
gender, and BMI, are mutually correlated, as shown next:

In [29]:
nhanes_df[["RIDAGEYR", "RIAGENDR", "BMXBMI"]].corr()

,RIDAGEYR,RIAGENDR,BMXBMI
RIDAGEYR,1.000000,-0.021398,0.023089
RIAGENDR,-0.021398,1.000000,0.080463
BMXBMI,0.023089,0.080463,1.000000


Although the correlations among these three variables are not strong, they are sufficient to induce fairly substantial differences in the regression coefficients (e.g. the gender coefficient changes from 3.23 to 3.58).  In this example, the gender effect becomes larger after we control for BMI - we can take this to mean that in the model fit above that did not include BMI, the BMI was masking part of the association between gender and blood pressure.  In other settings, including
additional covariates can reduce the association between a covariate and an outcome.

---

## 6. Visualization of Regression Models
